# Text-conditioned diffusion for Fashion-MNIST with classifier-free guidance

This notebook extends the score-based diffusion model of `fashion_mnist_diffusion.ipynb` so that it generates an image from a **text prompt**. Typing *"a flip-flop shoe worn to the beach"* should produce something that looks like a sandal, even though that phrase never appears in training.

Note: This notebook was completely written by Claude Code, prompted by me (JL) to follow the presentation of classifier-free guidance from my lecture.

**Running on Colab.** Choose *Runtime → Change runtime type* and pick a **TPU** or a **GPU** (T4 is fine). The first code cell finds the accelerator and, on a TPU runtime, installs PyTorch/XLA. Training takes roughly 10 minutes with a warm start and 20 minutes from scratch; see below.

### The idea

A caption $c$ is mapped to a vector by a frozen, pre-trained text encoder (the text half of **CLIP**). The network learns a *conditional* score $s_\theta(x,t;c) \approx \nabla \log p(x,t \mid c)$. With probability $p_{\text{uncond}} = 0.15$ the caption is replaced during training by a learned "no caption" vector $\varnothing$, so the same network also learns the *unconditional* score $s_\theta(x,t;\varnothing) \approx \nabla \log \rho(x,t)$.

At sampling time the two are combined, exactly as on the slide:
$$
\tilde s_w(x,t \mid c) \;=\; (1-w)\, s_\theta(x,t;\varnothing) \;+\; w\, s_\theta(x,t;c)
\;=\; s_\theta(x,t;\varnothing) + w\,\big(s_\theta(x,t;c) - s_\theta(x,t;\varnothing)\big).
$$
Here $w = 0$ is unconditional, $w = 1$ is plain conditional sampling, and $w > 1$ scales up the prompt-dependent part. The guided score is used in the backward drift $b_*(x,t) = b(x,t) - \nu\, \tilde s_w(x,t \mid c)$, and everything else is as in the first notebook.

### Where the captions come from

Fashion-MNIST has only 10 class labels. To teach the network about *regions* of text-embedding space rather than 10 isolated points, each class gets 25 hand-written captions (5 descriptions × 5 templates, such as *"a black and white photo of a gladiator sandal"*). A separate list of **held-out prompts** tests generalization; a cell checks that their key words never appear in the training captions.

### Changes from the first notebook

1. **Conditioning.** The caption embedding goes through a small MLP and is added to the time embedding inside every residual block. The last layer of that MLP starts at zero.
2. **Warm start (optional).** If `fashion_mnist_score_ema.pt` (saved by the first notebook) is present, the U-Net starts from those weights. Because the conditioning starts at zero, the initial network *is* the trained unconditional model, and training only has to learn the prompt-dependent part. Without the file, the network trains from scratch.
3. **Sampling with guidance** needs the conditional and unconditional scores at every step; both come from one forward pass on a doubled batch.
4. **A classifier "judge"**, a small CNN trained on real Fashion-MNIST images, measures how often samples land in the intended class.

In [ ]:
# Setup: detect the accelerator and, on a Colab TPU runtime, install PyTorch/XLA.
import glob, importlib.metadata, importlib.util, os, subprocess, sys

def tpu_present():
    return bool(glob.glob("/dev/accel*") or glob.glob("/dev/vfio/[0-9]*")
                or os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_ACCELERATOR_TYPE"))

if tpu_present() and importlib.util.find_spec("torch_xla") is None:
    try:
        torch_version = importlib.metadata.version("torch").split("+")[0]
        major_minor = ".".join(torch_version.split(".")[:2])
        packages = [f"torch=={torch_version}", f"torch_xla[tpu]=={major_minor}.*"]
    except importlib.metadata.PackageNotFoundError:
        packages = ["torch", "torch_xla[tpu]"]
    print("TPU detected; installing", " ".join(packages))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages,
                    "-f", "https://storage.googleapis.com/libtpu-releases/index.html"], check=True)

if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)

In [ ]:
import copy, math, time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.utils import make_grid
from transformers import CLIPTextModelWithProjection, CLIPTokenizer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()   # hide the report about CLIP's unused image-encoder weights

if torch.cuda.is_available():
    BACKEND, device = "cuda", torch.device("cuda")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
elif importlib.util.find_spec("torch_xla") is not None and tpu_present():
    import torch_xla
    import torch_xla.core.xla_model as xm
    BACKEND = "xla"
    device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
elif torch.backends.mps.is_available():
    BACKEND, device = "mps", torch.device("mps")
else:
    BACKEND, device = "cpu", torch.device("cpu")


def sync():
    # On a TPU, PyTorch/XLA records operations lazily; this runs the recorded graph.
    # On other devices it does nothing.
    if BACKEND == "xla":
        torch_xla.sync() if hasattr(torch_xla, "sync") else xm.mark_step()


print(f"Using {BACKEND} ({device})")
if BACKEND == "cpu":
    print("No accelerator found: training will be very slow. "
          "In Colab, choose Runtime -> Change runtime type -> GPU or TPU.")

## Settings

In [ ]:
# Forward SDE: dX_t = -THETA * X_t dt + sqrt(NU) dW_t on [T_MIN, T]  (same as the first notebook)
THETA = 0.5
NU = 1.0
T = 6.0
T_MIN = 1e-3

# Text encoder and classifier-free guidance
CLIP_MODEL = "openai/clip-vit-base-patch32"
EMB_DIM = 512          # size of a CLIP text embedding
P_UNCOND = 0.15        # probability of replacing the caption by the "no caption" vector

# Warm start from the first notebook (upload the file with the Files panel on the left)
WARM_START_PATH = "fashion_mnist_score_ema.pt"

# Training
BATCH_SIZE = 128
EPOCHS_WARM_START = 10
EPOCHS_FROM_SCRATCH = 20
LR = 2e-4
EMA_DECAY = 0.999
SEED = 0

# Sampling
N_SAMPLE_STEPS = 500
SAMPLE_BATCH = 80      # images are generated in chunks of this size, so a TPU compiles once
GUIDANCE = 3.0         # default guidance scale w

DATA_DIR = "./data"
torch.manual_seed(SEED)
np.random.seed(SEED)

## The forward process

As in the first notebook: $X_t \mid X_0 = z \sim N(\alpha_t z, \beta_t^2 I)$ with $\alpha_t = e^{-\theta t}$ and $\beta_t^2 = \frac{\nu}{2\theta}(1 - e^{-2\theta t})$.

In [ ]:
def alpha_beta(t):
    alpha = torch.exp(-THETA * t)
    beta = torch.sqrt(NU / (2 * THETA) * -torch.expm1(-2 * THETA * t))
    return alpha, beta


def drift(x, t):
    return -THETA * x


def bcast(v):
    # Reshape a per-example vector of shape (B,) to (B, 1, 1, 1) so it broadcasts over images.
    return v[:, None, None, None]

## Data

This time we keep the labels $y_i$ as well as the images $z_i$. The label is used only to pick a caption for the image; the network never sees it directly.

In [ ]:
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

train_set = torchvision.datasets.FashionMNIST(root=DATA_DIR, train=True, download=True)
test_set = torchvision.datasets.FashionMNIST(root=DATA_DIR, train=False, download=True)


def to_images(ds):
    return ds.data.float().div(127.5).sub(1.0).unsqueeze(1)   # (n, 1, 28, 28), values in [-1, 1]


Z, Y = to_images(train_set), train_set.targets.clone()
Z_test, Y_test = to_images(test_set), test_set.targets.clone()
if BACKEND == "cuda":
    Z, Y = Z.pin_memory(), Y.pin_memory()
n = Z.shape[0]
print("training images:", tuple(Z.shape))


def show_rows(images, row_labels, ncol, title=None, ax=None, fontsize=9):
    # Show images in rows of length ncol, with a label to the left of each row.
    grid = make_grid((images.float().cpu() + 1) / 2, nrow=ncol, padding=2, pad_value=1.0)
    ax = ax or plt.gca()
    ax.imshow(grid.permute(1, 2, 0).clamp(0, 1), cmap="gray")
    ax.set_xticks([])
    ax.set_yticks([2 + 30 * i + 14 for i in range(len(row_labels))])
    ax.set_yticklabels(row_labels, fontsize=fontsize)
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    if title:
        ax.set_title(title)


examples = torch.cat([Z[Y == k][:10] for k in range(10)])
plt.figure(figsize=(6, 6))
show_rows(examples, CLASS_NAMES, ncol=10, title="Training images by class")
plt.show()

## Captions

Each class has 5 descriptions, and each description goes into 5 templates, for 250 training captions in all. During training, an image of class $k$ is paired with a caption drawn at random from class $k$'s 25 captions.

The **held-out prompts** are used only for testing. Their key words (listed below) appear in no training caption, which the cell checks. Some words are shared on purpose: *"shoe"* appears in both sandal and sneaker captions, and *"shirt"* in both T-shirt and Shirt captions, so they don't give away the answer.

In [ ]:
DESCRIPTORS = {
    "T-shirt/top": ["a t-shirt", "a tee shirt", "a short-sleeved top", "a tank top", "a crew neck t-shirt"],
    "Trouser":     ["a pair of trousers", "a pair of pants", "a pair of slacks", "long pants", "tailored trousers"],
    "Pullover":    ["a pullover", "a sweater", "a knit jumper", "a sweatshirt", "a long-sleeved knit top"],
    "Dress":       ["a dress", "a summer dress", "an evening gown", "a maxi dress", "a cocktail dress"],
    "Coat":        ["a coat", "an overcoat", "a trench coat", "a parka", "a raincoat"],
    "Sandal":      ["a sandal", "a strappy sandal", "an open-toed shoe", "a high-heeled sandal", "a gladiator sandal"],
    "Shirt":       ["a shirt", "a button-down shirt", "a dress shirt", "a collared shirt", "a blouse"],
    "Sneaker":     ["a sneaker", "a trainer", "an athletic shoe", "a canvas shoe", "a low-top sneaker"],
    "Bag":         ["a bag", "a handbag", "a purse", "a tote bag", "a backpack"],
    "Ankle boot":  ["an ankle boot", "a boot", "a chelsea boot", "a short leather boot", "a heeled bootie"],
}
TEMPLATES = ["{}", "a photo of {}", "a black and white photo of {}", "a picture of {}",
             "a low-resolution image of {}"]

CAPTIONS, CAPTION_CLASS = [], []
for k, name in enumerate(CLASS_NAMES):
    for d in DESCRIPTORS[name]:
        for template in TEMPLATES:
            CAPTIONS.append(template.format(d))
            CAPTION_CLASS.append(k)
CAPTION_CLASS = torch.tensor(CAPTION_CLASS)

# Held-out prompts and the class we hope each one produces (None = off-domain).
HELD_OUT = [
    ("a flip-flop shoe worn to the beach", "Sandal"),
    ("running shoes for the gym", "Sneaker"),
    ("hiking footwear for muddy trails", "Ankle boot"),
    ("something to carry groceries in", "Bag"),
    ("denim jeans", "Trouser"),
    ("a warm winter jacket", "Coat"),
    ("a cozy cardigan for cold days", "Pullover"),
    ("something to wear to a fancy party", "Dress"),
    ("what a man wears to the office under his suit", "Shirt"),
    ("a plain undershirt", "T-shirt/top"),
    ("a golden retriever", None),
    ("a red sports car", None),
]
HELD_OUT_WORDS = ["flip", "flop", "beach", "running", "gym", "hiking", "muddy", "trails",
                  "carry", "groceries", "denim", "jeans", "warm", "winter", "jacket", "cozy",
                  "cardigan", "cold", "fancy", "party", "office", "suit", "undershirt", "plain"]

leaks = [(word, cap) for word in HELD_OUT_WORDS for cap in CAPTIONS if word in cap.lower()]
assert not leaks, f"held-out words found in training captions: {leaks}"
print(f"{len(CAPTIONS)} training captions; no held-out word appears in any of them.")
print("Some examples:")
for i in np.random.default_rng(0).choice(len(CAPTIONS), 8, replace=False):
    print(f"  [{CLASS_NAMES[CAPTION_CLASS[i]]:>11}]  {CAPTIONS[i]}")

## The text encoder

We use the text half of CLIP, a model trained to match 400 million images with their captions, so its embeddings reflect what things *look like*. It is frozen and runs on the CPU. Every training caption is embedded **once**, and only the resulting $250 \times 512$ table goes to the accelerator. Embeddings are normalized to unit length.

How a held-out prompt will behave depends mostly on where CLIP puts it. The heat map compares each prompt with the average embedding of each class's training captions, converted to probabilities the way CLIP does (a softmax of $100 \times$ cosine similarity). The box marks the class we hope for.

Not every prompt lands in its box. For example, CLIP associates *"under his suit"* with trousers. The diffusion model can only follow the embedding it is given, so these prompts are not expected to give the boxed class, however well the network is trained.

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(CLIP_MODEL)
text_encoder = CLIPTextModelWithProjection.from_pretrained(CLIP_MODEL).eval()   # stays on the CPU


@torch.no_grad()
def embed_text(texts):
    tokens = tokenizer(list(texts), padding=True, return_tensors="pt")
    return F.normalize(text_encoder(**tokens).text_embeds, dim=-1)


E_train = embed_text(CAPTIONS)                                       # (250, 512)
class_emb = F.normalize(torch.stack([E_train[CAPTION_CLASS == k].mean(0) for k in range(10)]), dim=-1)


def class_probs(prompt_emb):
    # CLIP-style probabilities that each prompt "means" each class.
    return F.softmax(100.0 * prompt_emb @ class_emb.T, dim=-1)


held_out_prompts = [p for p, _ in HELD_OUT]
held_out_emb = embed_text(held_out_prompts)
P = class_probs(held_out_emb).numpy()

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(P, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=40, ha="right")
ax.set_yticks(range(len(HELD_OUT)))
ax.set_yticklabels(held_out_prompts)
for i in range(P.shape[0]):
    for j in range(P.shape[1]):
        if P[i, j] >= 0.05:
            ax.text(j, i, f"{100 * P[i, j]:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if P[i, j] > 0.5 else "black")
for i, (_, target) in enumerate(HELD_OUT):
    if target is not None:
        j = CLASS_NAMES.index(target)
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="red", lw=2))
ax.set_title("CLIP: held-out prompt vs. class (percent)")
plt.tight_layout()
plt.show()

## The score network

The U-Net is the same as in the first notebook, with one addition. The caption embedding $c$ goes through a small MLP, and the result is **added to the time embedding**, so it reaches every residual block. The last layer of the MLP starts at zero, so a freshly built network ignores the caption.

The "no caption" token $\varnothing$ is a learned 512-dimensional vector, `model.null`, that is fed through the same MLP.

The score is $s_\theta(x,t;c) = -\hat\epsilon_\theta(x,t;c) / \beta_t$.

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim, out_dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
        args = (1000.0 * t / T)[:, None] * freqs[None, :]
        return self.mlp(torch.cat([torch.sin(args), torch.cos(args)], dim=1))


class ResBlock(nn.Module):
    def __init__(self, c_in, c_out, t_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, c_in)
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.time = nn.Linear(t_dim, c_out)
        self.norm2 = nn.GroupNorm(8, c_out)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.skip = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time(F.silu(t_emb))[:, :, None, None]
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.skip(x)


class SelfAttention(nn.Module):
    def __init__(self, channels, heads=4):
        super().__init__()
        self.heads = heads
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, 3 * channels, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(self.norm(x)).reshape(B, 3, self.heads, C // self.heads, H * W)
        q, k, v = (u.transpose(-1, -2) for u in qkv.unbind(1))   # each (B, heads, HW, C/heads)
        h = F.scaled_dot_product_attention(q, k, v)
        return x + self.proj(h.transpose(-1, -2).reshape(B, C, H, W))


class UNet(nn.Module):
    def __init__(self, ch=(64, 128, 256), t_dim=256, emb_dim=EMB_DIM):
        super().__init__()
        c1, c2, c3 = ch
        self.time = TimeEmbedding(c1, t_dim)
        # New: caption conditioning, zero at initialization, and the "no caption" vector.
        self.cond = nn.Sequential(nn.Linear(emb_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        nn.init.zeros_(self.cond[-1].weight)
        nn.init.zeros_(self.cond[-1].bias)
        self.null = nn.Parameter(torch.zeros(emb_dim))

        self.inc = nn.Conv2d(1, c1, 3, padding=1)
        self.down1 = ResBlock(c1, c1, t_dim)                        # 28 x 28
        self.pool1 = nn.Conv2d(c1, c1, 3, stride=2, padding=1)     # -> 14 x 14
        self.down2 = ResBlock(c1, c2, t_dim)
        self.pool2 = nn.Conv2d(c2, c2, 3, stride=2, padding=1)     # -> 7 x 7
        self.down3 = ResBlock(c2, c3, t_dim)
        self.mid1 = ResBlock(c3, c3, t_dim)
        self.attn = SelfAttention(c3)
        self.mid2 = ResBlock(c3, c3, t_dim)
        self.up3 = ResBlock(2 * c3, c3, t_dim)
        self.unpool2 = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                     nn.Conv2d(c3, c3, 3, padding=1))   # -> 14 x 14
        self.up2 = ResBlock(c3 + c2, c2, t_dim)
        self.unpool1 = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                     nn.Conv2d(c2, c2, 3, padding=1))   # -> 28 x 28
        self.up1 = ResBlock(c2 + c1, c1, t_dim)
        self.out = nn.Sequential(nn.GroupNorm(8, c1), nn.SiLU(), nn.Conv2d(c1, 1, 3, padding=1))
        nn.init.zeros_(self.out[-1].weight)
        nn.init.zeros_(self.out[-1].bias)

    def forward(self, x, t, c):
        e = self.time(t) + self.cond(c)          # c: (B, 512) caption embeddings, or the null vector
        h1 = self.down1(self.inc(x), e)
        h2 = self.down2(self.pool1(h1), e)
        h3 = self.down3(self.pool2(h2), e)
        h = self.mid2(self.attn(self.mid1(h3, e)), e)
        h = self.up3(torch.cat([h, h3], dim=1), e)
        h = self.up2(torch.cat([self.unpool2(h), h2], dim=1), e)
        h = self.up1(torch.cat([self.unpool1(h), h1], dim=1), e)
        return self.out(h)


model = UNet()
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

### Warm start

If the first notebook's weights are available, load them. Only the new conditioning layers and the null vector are missing from that file, and they start at zero, so at step 0 the network computes exactly the unconditional score from the first notebook for every caption.

To use the warm start on Colab, upload `fashion_mnist_score_ema.pt` with the Files panel on the left (or copy it from Google Drive) before running this cell.

In [ ]:
if os.path.exists(WARM_START_PATH):
    state = torch.load(WARM_START_PATH, map_location="cpu")
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert not unexpected, f"unexpected keys in checkpoint: {unexpected}"
    assert all(k.startswith(("cond.", "null")) for k in missing), f"missing keys: {missing}"
    EPOCHS = EPOCHS_WARM_START
    print(f"Warm start from {WARM_START_PATH}; new parameters: {missing}")
else:
    EPOCHS = EPOCHS_FROM_SCRATCH
    print(f"{WARM_START_PATH} not found; training from scratch.")
print(f"Training for {EPOCHS} epochs.")

model = model.to(device)
ema = copy.deepcopy(model).eval()
for p in ema.parameters():
    p.requires_grad_(False)

## Training with classifier-free guidance

This is the algorithm from the first notebook with captions added. For each random mini-batch $\mathcal B$, and for each $i \in \mathcal B$:

- sample $t_i \sim \text{Unif}[t_{\min}, T]$ and $\epsilon_i \sim N(0, I_d)$, and set $x_i = \alpha_{t_i} z_i + \beta_{t_i}\, \epsilon_i$,
- pick a random training caption for class $y_i$ and let $c_i$ be its CLIP embedding,
- with probability $p_{\text{uncond}}$, replace $c_i$ by $\varnothing$.

Then take a gradient step on
$$
\mathcal L_{\mathcal B}(\theta) = \frac{1}{|\mathcal B|\, d} \sum_{i \in \mathcal B} \big\| \hat\epsilon_\theta(x_i, t_i; c_i) - \epsilon_i \big\|^2 .
$$
The examples with $c_i = \varnothing$ are ordinary unconditional score matching, so the one network learns both $s_\theta(x,t;c)$ and $s_\theta(x,t;\varnothing)$.

On a TPU, the first step takes a minute or two while XLA compiles the training step.

In [ ]:
E_train_dev = E_train.to(device)
caps_by_class = torch.stack([torch.nonzero(CAPTION_CLASS == k).squeeze(1) for k in range(10)]).to(device)
N_PER_CLASS = caps_by_class.shape[1]          # 25 captions per class

optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def caption_batch(y):
    # A random training caption for each label, replaced by the null vector with probability P_UNCOND.
    # torch.where keeps every tensor the same shape, so a TPU does not recompile.
    B = y.shape[0]
    k = torch.randint(0, N_PER_CLASS, (B,), device=device)
    c = E_train_dev[caps_by_class[y, k]]
    drop = torch.rand(B, device=device) < P_UNCOND
    return torch.where(drop[:, None], model.null.expand(B, -1), c)


def train_step(z, y, ema_decay):
    B = z.shape[0]
    t = T_MIN + (T - T_MIN) * torch.rand(B, device=device)     # t_i ~ Unif[T_MIN, T]
    eps = torch.randn_like(z)                                    # eps_i ~ N(0, I_d)
    alpha, beta = alpha_beta(t)
    x = bcast(alpha) * z + bcast(beta) * eps                     # x_i = alpha_{t_i} z_i + beta_{t_i} eps_i
    c = caption_batch(y)                                         # c_i, or the null vector
    loss = F.mse_loss(model(x, t, c), eps)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    with torch.no_grad():
        for p_ema, p in zip(ema.parameters(), model.parameters()):
            p_ema.lerp_(p, 1.0 - ema_decay)   # ema_decay is a tensor, so a TPU does not recompile
    return loss.detach()


steps_per_epoch = n // BATCH_SIZE        # drop the last partial batch so every batch has the same shape
LOG_EVERY = 100
loss_log = []
running = torch.zeros((), device=device)
step = 0
start = time.time()
for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(n)
    for k in range(steps_per_epoch):
        batch = perm[k * BATCH_SIZE:(k + 1) * BATCH_SIZE]        # a random mini-batch B
        z = Z[batch].to(device, non_blocking=True)
        y = Y[batch].to(device, non_blocking=True)
        ema_decay = torch.tensor(min(EMA_DECAY, (1 + step) / (10 + step))).to(device)
        running = running + train_step(z, y, ema_decay)
        sync()
        step += 1
        if step % LOG_EVERY == 0:
            loss_log.append(running.item() / LOG_EVERY)
            running = torch.zeros((), device=device)
    print(f"epoch {epoch + 1:3d}/{EPOCHS}   loss {loss_log[-1]:.4f}   "
          f"elapsed {(time.time() - start) / 60:5.1f} min")

torch.save({k: v.cpu() for k, v in ema.state_dict().items()}, "fashion_mnist_cfg_ema.pt")

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(np.arange(1, len(loss_log) + 1) * LOG_EVERY, loss_log)
plt.xlabel("training step")
plt.ylabel("loss")
plt.yscale("log")
plt.title(f"Training loss (average over {LOG_EVERY} steps)")
plt.grid(alpha=0.3)
plt.show()

## Sampling with guidance

The reverse-time Euler–Maruyama steps are as in the first notebook, with the guided score in place of $s_\theta(x,t)$:
$$
X_{t-h} = X_t - h\, b_*(X_t, t) + \sqrt{\nu h}\; Z, \qquad
b_*(x,t) = b(x,t) - \nu\, \tilde s_w(x,t \mid c), \qquad
\tilde s_w = (1-w)\, s_\theta(x,t;\varnothing) + w\, s_\theta(x,t;c).
$$
Both scores are needed at every step. Rather than two forward passes, we stack the batch with itself, giving the first copy the caption and the second copy $\varnothing$, and split the output.

`generate` takes one caption embedding and one guidance scale $w$ per image, so a single call can mix prompts and scales.

In [ ]:
def guided_score(net, x, t, c_and_null, w):
    # One forward pass on the doubled batch [x; x] with captions [c; null].
    eps = net(torch.cat([x, x]), torch.cat([t, t]), c_and_null)
    eps_c, eps_null = eps.chunk(2)
    _, beta = alpha_beta(t)
    eps_guided = (1 - bcast(w)) * eps_null + bcast(w) * eps_c
    return -eps_guided / bcast(beta)          # s~_w = (1 - w) s(x,t;null) + w s(x,t;c)


@torch.no_grad()
def sample_chunk(net, c, w, n_steps):
    m = c.shape[0]
    times = np.geomspace(T, T_MIN, n_steps + 1)       # uniform in log t, small steps near t = 0
    c_and_null = torch.cat([c, net.null.expand(m, -1)])
    x = math.sqrt(NU / (2 * THETA)) * torch.randn(m, 1, 28, 28, device=device)
    for i in range(n_steps):
        t = torch.full((m,), float(times[i])).to(device)
        h = torch.full((m,), float(times[i] - times[i + 1])).to(device)
        b_star = drift(x, t) - NU * guided_score(net, x, t, c_and_null, w)
        x = x - bcast(h) * b_star + bcast(torch.sqrt(NU * h)) * torch.randn_like(x)
        sync()
    t = torch.full((m,), float(times[-1])).to(device)
    alpha, beta = alpha_beta(t)
    z_hat = (x + bcast(beta) ** 2 * guided_score(net, x, t, c_and_null, w)) / bcast(alpha)  # Tweedie
    sync()
    return z_hat.clamp(-1, 1).cpu()


def generate(net, c, w, n_steps=N_SAMPLE_STEPS, seed=1):
    # c: (N, 512) caption embeddings; w: a number, or a length-N sequence of guidance scales.
    # Runs in chunks of exactly SAMPLE_BATCH (the last one padded), so a TPU compiles only once.
    torch.manual_seed(seed)
    net.eval()
    N = c.shape[0]
    w = torch.as_tensor(w, dtype=torch.float32).expand(N) if np.ndim(w) == 0 else torch.as_tensor(w, dtype=torch.float32)
    out = []
    for s in range(0, N, SAMPLE_BATCH):
        cc, ww = c[s:s + SAMPLE_BATCH], w[s:s + SAMPLE_BATCH]
        m = cc.shape[0]
        if m < SAMPLE_BATCH:
            cc = torch.cat([cc, cc[:1].expand(SAMPLE_BATCH - m, -1)])
            ww = torch.cat([ww, ww[:1].expand(SAMPLE_BATCH - m)])
        out.append(sample_chunk(net, cc.to(device), ww.to(device), n_steps)[:m])
    return torch.cat(out)

### One prompt per class, with and without guidance

Each row uses the first description of a class (a training caption), with 8 samples per row. On the left, $w = 1$ (plain conditional sampling); on the right, $w = 3$.

In [ ]:
N_PER_ROW = 8
class_prompts = [DESCRIPTORS[name][0] for name in CLASS_NAMES]
c = embed_text(class_prompts).repeat_interleave(N_PER_ROW, dim=0)

start = time.time()
samples = generate(ema, torch.cat([c, c]), [1.0] * len(c) + [GUIDANCE] * len(c))
print(f"sampling took {time.time() - start:.1f} s")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
labels = [f'"{p}"' for p in class_prompts]
show_rows(samples[:len(c)], labels, N_PER_ROW, title="w = 1 (conditional)", ax=axes[0])
show_rows(samples[len(c):], labels, N_PER_ROW, title=f"w = {GUIDANCE:g} (guided)", ax=axes[1])
plt.tight_layout()
plt.show()

### The effect of the guidance scale

One held-out prompt at several values of $w$. At $w = 0$ the prompt is ignored. As $w$ grows the images match the prompt more consistently and look more alike: guidance trades diversity for fidelity.

In [ ]:
SWEEP_PROMPT = "a flip-flop shoe worn to the beach"
SWEEP_W = [0.0, 1.0, 2.0, 4.0, 7.0]

c = embed_text([SWEEP_PROMPT]).expand(len(SWEEP_W) * N_PER_ROW, -1)
w = np.repeat(SWEEP_W, N_PER_ROW)
samples = generate(ema, c, w)

plt.figure(figsize=(7, 5))
show_rows(samples, [f"w = {v:g}" for v in SWEEP_W], N_PER_ROW, title=f'"{SWEEP_PROMPT}"')
plt.show()

## A classifier as judge

To measure whether samples match their prompts, we train a small CNN on the real Fashion-MNIST training images (about a minute) and check it on the test set. It then labels generated images.

In [ ]:
class Judge(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                                       # -> 14 x 14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                                       # -> 7 x 7
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.net(x)


judge_net = Judge().to(device)
judge_opt = torch.optim.Adam(judge_net.parameters(), lr=1e-3)
JUDGE_EPOCHS, JUDGE_BATCH = 4, 250


def judge_step(x, y):
    loss = F.cross_entropy(judge_net(x), y)
    judge_opt.zero_grad(set_to_none=True)
    loss.backward()
    judge_opt.step()


start = time.time()
judge_net.train()
for epoch in range(JUDGE_EPOCHS):
    perm = torch.randperm(n)
    for k in range(n // JUDGE_BATCH):
        batch = perm[k * JUDGE_BATCH:(k + 1) * JUDGE_BATCH]
        judge_step(Z[batch].to(device), Y[batch].to(device))
        sync()
judge_net.eval()


@torch.no_grad()
def judge(images):
    # Predicted class of each image, computed in chunks of SAMPLE_BATCH (the last one padded).
    preds = []
    for s in range(0, len(images), SAMPLE_BATCH):
        x = images[s:s + SAMPLE_BATCH]
        m = x.shape[0]
        if m < SAMPLE_BATCH:
            x = torch.cat([x, x[:1].expand(SAMPLE_BATCH - m, -1, -1, -1)])
        logits = judge_net(x.to(device))
        sync()
        preds.append(logits.argmax(1).cpu()[:m])
    return torch.cat(preds)


test_acc = (judge(Z_test) == Y_test).float().mean().item()
print(f"judge trained in {time.time() - start:.0f} s; test accuracy {100 * test_acc:.1f}%")

### Held-out prompts

For each held-out prompt: CLIP's top three classes, 8 samples at $w = 3$, and the judge's verdict. The last two prompts are off-domain. The model can only draw clothing, so it draws whatever item CLIP thinks is closest.

In [ ]:
c = held_out_emb.repeat_interleave(N_PER_ROW, dim=0)
samples = generate(ema, c, GUIDANCE)
preds = judge(samples).view(len(HELD_OUT), N_PER_ROW)
probs = class_probs(held_out_emb)

fig, axes = plt.subplots(len(HELD_OUT), 2, figsize=(11, 1.25 * len(HELD_OUT)),
                         gridspec_kw={"width_ratios": [1.15, 2]})
for i, (prompt, target) in enumerate(HELD_OUT):
    top = probs[i].topk(3)
    clip_txt = ", ".join(f"{CLASS_NAMES[j]} {100 * p:.0f}%" for p, j in zip(top.values, top.indices))
    counts = torch.bincount(preds[i], minlength=10)
    if target is None:
        j = counts.argmax().item()
        verdict = f"judge: {counts[j]}/{N_PER_ROW} {CLASS_NAMES[j]}  (off-domain)"
    else:
        verdict = f"judge: {counts[CLASS_NAMES.index(target)]}/{N_PER_ROW} {target}"
    axes[i, 0].text(0, 0.5, f'"{prompt}"\nCLIP: {clip_txt}\n{verdict}', va="center", fontsize=9)
    axes[i, 0].axis("off")
    show_rows(samples[i * N_PER_ROW:(i + 1) * N_PER_ROW], [""], N_PER_ROW, ax=axes[i, 1])
plt.tight_layout()
plt.show()

### Accuracy as a function of $w$

For each guidance scale, we generate 12 images for each of the 10 class prompts and each of the 10 in-domain held-out prompts, and ask the judge how many land in the intended class. To save time this uses 250 sampling steps. Chance is 10%. The held-out curve can't reach 100%, because CLIP itself sends some held-out prompts to a different class (see the heat map).

In [ ]:
ACC_W = [0.0, 1.0, 2.0, 3.0, 5.0, 7.0]
N_ACC = 12
in_domain = [(p, CLASS_NAMES.index(t)) for p, t in HELD_OUT if t is not None]
prompt_sets = {
    "class prompts (training captions)": (embed_text(class_prompts), torch.arange(10)),
    "held-out prompts": (embed_text([p for p, _ in in_domain]), torch.tensor([k for _, k in in_domain])),
}

start = time.time()
accuracy = {}
for name, (emb, target) in prompt_sets.items():
    c = emb.repeat_interleave(N_ACC, dim=0).repeat(len(ACC_W), 1)
    w = np.repeat(ACC_W, len(emb) * N_ACC)
    preds = judge(generate(ema, c, w, n_steps=250, seed=2)).view(len(ACC_W), -1)
    correct = preds == target.repeat_interleave(N_ACC)[None, :]
    accuracy[name] = correct.float().mean(1).numpy()
print(f"took {(time.time() - start) / 60:.1f} min")

plt.figure(figsize=(6, 4))
for name, acc in accuracy.items():
    plt.plot(ACC_W, 100 * acc, "o-", label=name)
plt.axhline(10, color="gray", ls=":", label="chance")
plt.xlabel("guidance scale w")
plt.ylabel("judged correct (%)")
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.3)
plt.title("How often samples match the prompt")
plt.show()

## Try your own prompt

Edit the prompt and the guidance scale and rerun the cell.

In [ ]:
PROMPT = "a flip-flop shoe worn to the beach"
W = 3.0

emb = embed_text([PROMPT])
top = class_probs(emb)[0].topk(3)
print("CLIP's closest classes:",
      ", ".join(f"{CLASS_NAMES[j]} {100 * p:.0f}%" for p, j in zip(top.values, top.indices)))

samples = generate(ema, emb.expand(16, -1), W, seed=int(time.time()))
counts = torch.bincount(judge(samples), minlength=10)
print("judge:", ", ".join(f"{CLASS_NAMES[j]} {counts[j]}" for j in counts.argsort(descending=True) if counts[j] > 0))

plt.figure(figsize=(8, 2.4))
show_rows(samples, ["", ""], 8, title=f'"{PROMPT}"   (w = {W:g})')
plt.show()

## Things to try

- **The amount of caption dropout.** Retrain with `P_UNCOND = 0.05` and `0.5`. How do the $w = 0$ samples and the accuracy curve change?
- **Class names only.** Retrain with `DESCRIPTORS` reduced to one description per class and `TEMPLATES = ["{}"]`, so the network sees only 10 embedding vectors. How well do the held-out prompts work now?
- **Snap to the nearest caption.** Replace a prompt's embedding by the closest training caption embedding (`E_train[(emb @ E_train.T).argmax()]`) before sampling. When does this help, and what does it give up?
- **A different text encoder.** Swap CLIP for a sentence encoder such as `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions; set `EMB_DIM` to match). Does the heat map change? Do the held-out prompts?
- **Large guidance.** Push $w$ to 15 or 20. What happens to the images?